# Exercises week 43: reading a classifier — the confusion matrix, the ROC curve and the threshold

**FYS-STK3155/4155 Applied Data Analysis and Machine Learning**

**October 19–23, 2026**

**Deadline: Friday October 23 at midnight** (one notebook per group, in
Canvas).

## Aims of this week's exercises

Project 2 asks you to train classifiers — logistic regression as the
baseline and a neural network with hidden layers — and then to *say what
they do*: the confusion matrix on the test set, with its interpretation.
This week you build the tools for that, implementing them yourself from the
definitions and checking them against `scikit-learn`:

- the confusion matrix and the quantities read off its four counts —
  accuracy, precision, recall, $F_1$ (Section 5.12, Eqs. 5.67–5.70);
- the ROC curve and the area under it, the AUC, and why they measure
  something a single threshold cannot;
- (optional) the $K\times K$ confusion matrix of a multiclass classifier and
  how to read a $2\times2$ matrix off it.

The Monday lecture went through what every one of these measures *means* on
a worked screening example and then applied them to "is it a five?" on the
full MNIST set (`week43monday.pdf`, `week43monday.ipynb`); the Tuesday session
goes through the worked example in detail and then writes the same code on
the small digits set (`week43tuesday.ipynb`). The classifiers can be your
own Project 2 code (recommended: this is the code you will report on) or
`scikit-learn`'s `LogisticRegression` and `MLPClassifier`.

There is no validation set and no cross-validation in this week's
exercises: how to cross-validate your Project 2 network is the topic of next
week's Tuesday session and exercises.

**Reading:** Sections 5.12–5.13 of the lecture notes; Eq. (2.24) for the
standard error of a proportion. Hastie, Tibshirani and Friedman, Section
9.2.5; Raschka et al., Chapter 6 ("Looking at different performance
evaluation metrics").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics as skm
np.random.seed(2026)

## Exercise 1: your own confusion matrix, ROC curve and AUC, checked

Implement the following functions from their definitions. Labels are in
$\{0,1\}$ with $1$ the positive class; `scores` are the output probabilities
(or any monotone score) of a classifier.

**a)** `confusion_matrix(y_true, y_pred)` returning the four counts
`tp, fn, fp, tn` of Eq. (5.67), and from them the accuracy, precision,
recall and $F_1$ of Eqs. (5.68)–(5.70) (handle the cases where a
denominator is zero).

**b)** `roc_curve(y_true, scores)` returning the false positive rates, true
positive rates and thresholds for every distinct score, from **one sorted
pass**: sort by decreasing score and walk down the list — every sample
flagged adds one to TP if it is a positive and one to FP if it is a
negative; then `auc_trapezoid(fpr, tpr)` for the area.

**c)** `roc_auc(y_true, scores)` by the rank formula of the lecture,
$\mathrm{AUC}=\bigl(\sum_{i\in\mathrm{pos}}R_i-\tfrac12 n_+(n_++1)\bigr)/(n_+n_-)$,
with average ranks for tied scores (`np.unique` with `return_inverse` and
`return_counts`, or `scipy.stats.rankdata`). Explain in one sentence why
this is the probability that a random positive outscores a random negative.

**d)** Test everything on the small example below against `sklearn.metrics`
(`confusion_matrix`, `precision_score`, `recall_score`, `f1_score`,
`roc_curve` with `drop_intermediate=False`, `roc_auc_score`). Mind the
layout of `sklearn.metrics.confusion_matrix`: rows true $0/1$, columns
predicted $0/1$, so `[[TN, FP], [FN, TP]]`. The example has tied scores on
purpose: the three AUCs must still agree.

In [ ]:
def confusion_matrix(y_true, y_pred) -> tuple:
    """Return TP, FN, FP, TN for binary labels (0,1)."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fn, fp, tn


def classification_report(y_true, y_pred) -> dict:
    # accuracy, precision, recall and F1 from the four counts
    tp, fn, fp, tn = confusion_matrix(y_true, y_pred)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0.0)
    return dict(accuracy=accuracy, precision=precision, recall=recall, f1=f1)


def roc_curve(y_true, scores):
    """Calculate TPR, FPR and thresholds."""
    # sort by descending score
    order = np.argsort(scores, descending=True)
    sorted_scores = scores[order]
    sorted_labels = y_true[order]

    # count all positives and negatives
    n_positive = np.sum(y_true == 1) # tp + fn
    n_negative = np.sum(y_true == 0) # fp + tn

    # initialize counters and lists
    tp, fp = 0, 0
    thresholds = [np.inf]
    tpr = [0.0]
    fpr = [0.0]

    for i in range(len(sorted_scores)):
        score = sorted_scores[i]
        label = sorted_labels[i]

        # count tp encountered
        if label == 1:
            tp += 1
        else:
            fp += 1

        # record only one ROC point for each unique score
        if i == len(sorted_scores) - 1 or score != sorted_scores[i + 1]: # if at last score or if next score is different from current score
            tpr.append(tp / n_positive)
            fpr.append(fp / n_negative)
            thresholds.append(score)

    return np.array(fpr), np.array(tpr), np.array(thresholds)


def auc_trapezoid(fpr, tpr):
    """Calculate AUC using trapezoidial rule from NumPy (area under TPR curve)."""
    return np.trapezoid(tpr, fpr)


from scipy.stats import rankdata

def roc_auc(y_true, scores) -> float:
    """
    AUC measyres how often a positive sample receives a higher score
    than a negative sample
    """
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)

    # use SciPy to ...
    ranks = rankdata(scores, method="average")

    # sum ranks of positive samples only
    R_pos = np.sum(ranks[y_true == 1])

    auc = (R_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

    return auc


# a small test: 20 samples, 6 positives, scores rounded to one decimal so that there are ties
rng = np.random.default_rng(2)
y_small = np.array([1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0])
s_small = np.round(np.clip(0.35 * y_small + rng.normal(0.3, 0.2, 20), 0, 1), 1)
print(confusion_matrix(y_small, s_small > 0.5), skm.confusion_matrix(y_small, s_small > 0.5))
print(roc_auc(y_small, s_small), auc_trapezoid(*roc_curve(y_small, s_small)[:2]), skm.roc_auc_score(y_small, s_small))

We see that our implementation agrees with Scikit-learn. The AUC score measures how often a positive sample receives a higher score than a negative sample. If every positive score is higher than every negative score, AUC = 1. We got AUC = 0.857 which means that our classifier correctly ranks a randomly selected positive sample above a randomly selected negative sample $85.7\%$ of the time.

**The same with `scikit-learn`.** Once your functions agree with the
library, this is the template you can use in Project 2 for the test set
(`y_test` the true labels, `p_test` the output probabilities of your
classifier, `y_pred = p_test > 0.5` the decisions):

In [ ]:
from sklearn.metrics import (confusion_matrix as sk_confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score,
                             classification_report as sk_classification_report,
                             roc_curve as sk_roc_curve, roc_auc_score, RocCurveDisplay)

y_test, p_test = y_small, s_small          # replace by your test labels and output probabilities
y_pred = p_test > 0.5

cm = sk_confusion_matrix(y_test, y_pred)   # [[TN, FP], [FN, TP]]
print(cm)
print(f"accuracy {accuracy_score(y_test, y_pred):.4f}  precision {precision_score(y_test, y_pred, zero_division=0):.4f}  "
      f"recall {recall_score(y_test, y_pred):.4f}  F1 {f1_score(y_test, y_pred):.4f}")
print(sk_classification_report(y_test, y_pred, target_names=["negative", "positive"], zero_division=0))

fpr, tpr, thresholds = sk_roc_curve(y_test, p_test, drop_intermediate=False)
print(f"AUC {roc_auc_score(y_test, p_test):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"]).plot(ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, p_test, ax=ax[1], name="your classifier")
ax[1].plot([0, 1], [0, 1], "k--", lw=0.8)
plt.tight_layout(); plt.show()

## Exercise 2: MNIST, one digit against the rest

This is the Monday lecture's example on data of your choosing, with your
Project 2 code. Use the full MNIST set as described in Project 2 (the file
`mnist.npz`, 60 000 training and 10 000 test images, flattened and scaled to
$[0,1]$); if your laptop is slow, train on a subset of the training images
but evaluate on the full test set.

**a)** Pick a digit $d$ (the lecture used $5$; choose a different one) and
set $y=1$ for the images of $d$. Report the prevalence of $d$ in the test set
and the accuracy of the majority rule ("never $d$") — the number every
accuracy below must be compared with.

  - **ANSWER**:  we chose the digit 4 and there are $982$ images of the digit 4 in the MNIST test set among $10,000$ images. We get a prevalence of $982/10000 \approx 0.0982$ and an accuracy of $1-\text{prevalence} \approx 0.9018$, as printed by the code below. So about 1 in ten images are the digit 4. The accuracy is about $90\%$, which is the accuracy of a classifier that always predicts "not 4" (so it misses every actual 4). So it has a very high accuracy without even recognizing a single 4. 

**b)** Train logistic regression (no hidden layer, sigmoid output, cross
entropy) and a network with one hidden layer on the training set, with the
SGD of Project 2. On the test set, at threshold $0.5$, report for both the
confusion matrix in the layout of Eq. (5.67) and the accuracy, precision,
recall and $F_1$ from your own exercise-1 functions, checked against the
`scikit-learn` template. Say in two sentences what the matrices tell you
that the two accuracies do not, and what fraction of the remaining error
(relative to the majority rule) each classifier removed.

- **ANSWER:** the logistic regression model with no hidden layers performed a little worse than the neural network with one hidden layer (32 neurons). The accuracy increased from $98\%$ to $99\%$ for the NN and the recall increased from $88\%$ to $96\%$. There was also a very slight increase in the AUC. The confusion matrices tells us how many FN (how many 4's we missed) and how many FP (incorrectly classifed as 4). The accuracy does not tell us this, it only tells us the proportion o fthe data the model classified correctly, but not regarding the digit. The accuracy does not tell us how bad the model actually does, that's why we need the confusion matrix. 

**c)** The ROC curves of both classifiers in one figure, with the AUC by
your trapezoidal rule, by your rank formula and by `roc_auc_score`, and the
point corresponding to $t=0.5$ marked on each curve. Add a histogram of the
output probabilities of the test images, separately for the two classes,
on a logarithmic vertical axis. Where do the errors live, and what does the
difference between the two AUCs say that the confusion matrices at $0.5$ do
not?
- **ANSWER:** see plot under. We see from both plots that the NN is better at classifying the digit 4 than logistic regression is. The NN has fewer errors. The ROC curves show that both models perform well on the data set, but the NN curve is a bit closer to the upper left corner, that is, it achieves a higher TPR than logistic regression. Moreover, the three methods for compting AUC are basicly equal. From the histograms, we see that logistic regression has more overlap  between the two distrbutions, with some errors. The NN is more confident and assigns higher/lower porbabilities, so the two curves are more separated, with fewer predictions at the threshold. The NN has a slightly higher AUC so it is better than logistic regression at ranking the actual digit -4's above the not-4's. The confusion matrices only look at threshold 0.5.

**d)** Standard errors: of the test accuracy and of the recall (Eq. 2.24,
$\sqrt{\hat p(1-\hat p)/n}$ with the appropriate $n$ in each case). Would
you call the difference between your two classifiers significant?

- **ANSWER*:* Our code prints:

    Logistic regression:                     Neural network:

    Accuracy: 0.9827 +/- 0.0013              Accuracy: 0.9947 +/- 0.0007

    Recall:   0.8788 +/- 0.0104              Recall:   0.9593 +/- 0.0063

    So the NN has a lower standard error in both metrics. I would say this is significant as it is a significant imporvement. If we look at the differences between the accuracies and the recalls, the standard errors are much smaller, so the NN is more confident in a way. So i would call this a significant difference, although I woul prefer a longer analysis before making such a bold statement. 


In [ ]:
d = 4
data = np.load("mnist.npz")

x_train = data["x_train"].reshape(len(data["x_train"]), -1) / 255.0
x_test = data["x_test"].reshape(len(data["x_test"]), -1) / 255.0

# binary labels
y_train = (data["y_train"] == d).astype(float)
y_test = (data["y_test"] == d).astype(float)
print(f"Number of 4s in training set: {np.sum(data["y_train"] == d)}")
print(f"Number of 4s in test set: {np.sum(data["y_test"] == d)}")

prevalence = np.mean(y_test) # N images containing d / total N number of images
accuracy = 1 - prevalence # "never d"

print(f"Prevalence: {prevalence:.4f}")
print(f"Baseline accuracy: {accuracy:.4f}")

In [ ]:
# code from week 42:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

def ReLU(z):
    return jnp.where(z > 0, z, 0.0)

def ReLU_der(z):
    return jnp.where(z > 0, 1.0, 0.0)

def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def mse_der(predict, target):
    return 2 * (predict - target) / predict.size

def create_layers_batch(network_input_size, layer_output_sizes):
    layers = []
    i_size = network_input_size

    for i, layer_output_size in enumerate(layer_output_sizes):
        if i == len(layer_output_sizes) - 1:
            scale = np.sqrt(1 / i_size)  # Sigmoid output
        else:
            scale = np.sqrt(2 / i_size)  # ReLU hidden layers

        W = np.random.randn(i_size, layer_output_size) * scale
        b = np.zeros(layer_output_size)

        layers.append((W, b))
        i_size = layer_output_size
    return layers

def feed_forward_batch(input, layers, activation_funcs):
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        z = a @ W + b
        a = activation_func(z)
    return a

def feed_forward_saver_batch(input, layers, activation_funcs):
    layer_inputs = []
    zs = []
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        layer_inputs.append(a)
        z = a @ W + b
        a = activation_func(z)

        zs.append(z)

    return layer_inputs, zs, a

def backpropagation_batch(input, layers, activation_funcs, target, activation_ders, cost_der=mse_der):
    layer_inputs, zs, predict = feed_forward_saver_batch(input, layers, activation_funcs)

    layer_grads = [() for layer in layers]

    # We loop over the layers, from the last to the first
    for i in reversed(range(len(layers))):
        layer_input, z, activation_der = layer_inputs[i], zs[i], activation_ders[i]

        if i == len(layers) - 1:
            if cost_der is None:
                # Softmax output + cross entropy: the derivatives combine to dC_dz(L) = prediction - target (Eq. 8.69)
                dC_dz = predict - target
            else:
                # For last layer we use cost derivative as dC_da(L) can be computed directly
                dC_dz = cost_der(predict, target) * activation_der(z)
        else:
            # For other layers we build on previous z derivative, as dC_da(i) = dC_dz(i+1) * dz(i+1)_da(i)
            (W, b) = layers[i + 1]
            dC_dz = (dC_dz @ W.T) * activation_der(z)

        dC_dW = layer_input.T @ dC_dz / input.shape[0]
        dC_db = np.mean(dC_dz, axis=0)

        layer_grads[i] = (dC_dW, dC_db)

    return layer_grads

def train_network_sgd_momentum(
    inputs, layers, activation_funcs, activation_ders, targets,
    gamma=0.001, beta=0.9, epochs=100, batch_size=16, seed=2026
):
    rng = np.random.default_rng(seed)
    n = inputs.shape[0]
    velocity = [(np.zeros_like(W), np.zeros_like(b)) for W, b in layers]

    for epoch in range(epochs):
        # Shuffle the data and loop over minibatches, one update per minibatch
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = perm[start : start + batch_size]
            layers_grad = backpropagation_batch(
                inputs[idx], layers, activation_funcs, targets[idx], activation_ders, cost_der=None
            )
            # Momentum: v = beta * v + gamma * gradient, theta = theta - v
            velocity = [
                (beta * v_W + gamma * W_g, beta * v_b + gamma * b_g)
                for (v_W, v_b), (W_g, b_g) in zip(velocity, layers_grad)
            ]
            layers = [(W - v_W, b - v_b) for (W, b), (v_W, v_b) in zip(layers, velocity)]
    return layers

In [ ]:
# Exercise 2b):

# Logistic regression: no hidden layers

np.random.seed(2026)

# 784 input features (pixels) and one output neuron
layers_logreg = create_layers_batch(784, [1])

# sigmoid output for binary classification
activation_funcs = [sigmoid]
activation_ders = [None]

# reshape targets to match the network output
targets_train = y_train.reshape(-1, 1)

# train
layers_logreg = train_network_sgd_momentum(
    inputs=x_train,
    layers=layers_logreg,
    activation_funcs=activation_funcs,
    activation_ders=activation_ders,
    targets=targets_train,
    gamma=0.01,
    beta=0.9,
    epochs=20,
    batch_size=64,
    seed=2026
)

# predict probabilities and classify using threshold 0.5
p_logreg = np.asarray(feed_forward_batch(x_test, layers_logreg, activation_funcs)).ravel()
y_pred_logreg = (p_logreg > 0.5).astype(int)


# Neural network: one hidden layer

np.random.seed(2026)

# 784 inputs, 32 hidden neurons and one output neuron
layers_nn = create_layers_batch(784, [32, 1])

# ReLU in hidden layer, sigmoid in output layer
activation_funcs_nn = [ReLU, sigmoid]
activation_ders_nn = [ReLU_der, None]

# train using the same optimizer and hyperparameters
layers_nn = train_network_sgd_momentum(
    inputs=x_train,
    layers=layers_nn,
    activation_funcs=activation_funcs_nn,
    activation_ders=activation_ders_nn,
    targets=targets_train,
    gamma=0.01,
    beta=0.9,
    epochs=20,
    batch_size=64,
    seed=2026
)

# Predict probabilities and classify using threshold 0.5
p_nn = np.asarray(feed_forward_batch(x_test, layers_nn, activation_funcs_nn)).ravel()
y_pred_nn = (p_nn > 0.5).astype(int)


# Compare the two classifiers and with Scikit-Learn:

baseline_accuracy = 1 - np.mean(y_test)
baseline_error = 1 - baseline_accuracy

from sklearn.metrics import confusion_matrix as sk_confusion_matrix
from sklearn.metrics import classification_report as sk_classification_report

for name, y_pred, probabilities in [
    ("Logistic regression", y_pred_logreg, p_logreg),
    ("Neural network", y_pred_nn, p_nn)
]:
    print(f"\n{name}:")

    tp, fn, fp, tn = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    conf_matrix = np.array([[tn, fp], [fn, tp]])

    print("Our confusion matrix:")
    print(conf_matrix)

    print("\nOur classification report:")
    print(report)

    # scikit-learn
    print("\nScikit-learn confusion matrix:")
    print(sk_confusion_matrix(y_test, y_pred))
    print("\nScikit-learn classification report:")
    print(sk_classification_report(y_test, y_pred, digits=4, zero_division=0))

    # check fraction of error removed
    baseline_error = np.mean(y_test)
    model_error = 1 - report["accuracy"]
    error_removed = (baseline_error - model_error) / baseline_error
    print(f"Baseline error removed: {error_removed:.2%}")

In [ ]:
# exercise 2c)

from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

for name, probabilities, color in [
    ("Logistic regression:", p_logreg, "#2F8FC4"),
    ("Neural network:", p_nn, "#C94B4B")
]:
    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, probabilities)

    # compare the three AUC calculations
    auc_trap = auc_trapezoid(fpr, tpr)
    auc_rank = roc_auc(y_test, probabilities)
    auc_sk = roc_auc_score(y_test, probabilities)

    print(f"\n{name}")
    print(f"Trapezoidal AUC:  {auc_trap:.6f}")
    print(f"Rank-based AUC:   {auc_rank:.6f}")
    print(f"Scikit-learn AUC: {auc_sk:.6f}")

    plt.plot(fpr, tpr, color=color, label=f"{name} (AUC = {auc_rank:.4f})")

    # Point corresponding to threshold t = 0.5
    y_pred = (probabilities > 0.5).astype(int)

    tp, fn, fp, tn = confusion_matrix(y_test, y_pred)

    fpr_05 = fp / (fp + tn)
    tpr_05 = tp / (tp + fn)

    plt.scatter(fpr_05, tpr_05, color=color, s=60, edgecolors="black", zorder=5)

plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random classifier")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves for MNIST digit 4")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# chatgpt assisted in making these histograms:

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
bins = np.linspace(0, 1, 51)

for ax, name, probabilities in [
    (axes[0], "Logistic regression", p_logreg),
    (axes[1], "Neural network", p_nn)
]:
    # Separate predictions by their actual class
    ax.hist(probabilities[y_test == 0], bins=bins, alpha=0.6, label="Not 4", color="#2F8FC4")
    ax.hist(probabilities[y_test == 1], bins=bins, alpha=0.6, label="Digit 4", color="#C94B4B")
    ax.axvline(0.5, color="black", linestyle="--", label="Threshold = 0.5")
    ax.set_yscale("log")
    ax.set_xlabel("Predicted probability of digit 4")
    ax.set_title(name)
    ax.legend()

axes[0].set_ylabel("Number of images")

plt.tight_layout()
plt.show()

In [ ]:
# Exercise 2d)

n_test = len(y_test)
n_positive = np.sum(y_test == 1)

for name, y_pred in [
    ("Logistic regression", y_pred_logreg),
    ("Neural network", y_pred_nn)
]:
    report = classification_report(y_test, y_pred)
    accuracy = report["accuracy"]
    recall = report["recall"]

    standard_error_accuracy = np.sqrt(accuracy * (1 - accuracy) / n_test) # uses test images
    standard_error_recall = np.sqrt(recall * (1 - recall) / n_positive) # uses actual images of 4

    print(f"\n{name}")
    print(f"Accuracy: {accuracy:.4f} +/- {standard_error_accuracy:.4f}")
    print(f"Recall:   {recall:.4f} +/- {standard_error_recall:.4f}")

## Exercise 3 (optional): ten classes — the matrix Project 2 asks for

**a)** Train your softmax network (or `scikit-learn`'s `MLPClassifier`, or
the week-41 `[784, 100, 10]` network) on the MNIST training set and build
the $10\times10$ confusion matrix of the test set by hand ($C_{ij}$ = number
of test images of true class $i$ predicted as class $j$; `np.add.at` does it
in one line). Check it against `sklearn.metrics.confusion_matrix`.

**b)** From the matrix: the per-class precision (columns) and recall (rows),
checked against `sklearn.metrics.classification_report`; the five most
frequent confusions (true $\to$ predicted); the $2\times2$ matrix of your
digit $d$ against the rest, read off the big matrix. Compare the latter with
the binary classifiers of exercise 2: same model? same decision rule? which
is better, and why?

**c)** Plot the matrix as an image with the diagonal removed, annotated with
the counts, and the per-class precision and recall as bars. Which pairs of
digits are confused in *both* directions, and which only in one? Look at a
few of the confused images and say whether you would have got them right.

**d)** The test accuracy comes with the standard error of Eq. (2.24). Train
a second network with a different architecture or a different seed and
report both accuracies with their error bars. Is the difference
significant? How many test images would you need to resolve a difference of
$0.1$ percentage points at $97\%$ accuracy?

In [ ]:
...

## Insights you should have gained this week — a self-check

Before you hand in, go through the list below without looking at the notes.
Each row is a question we may ask in the Tuesday session or on Project 2;
the right-hand column is the insight it tests, in one sentence.

| Ask yourself | The insight it tests |
|---|---|
| A classifier scores $97\%$ accuracy. Is it good? | Not knowable without the majority-class accuracy: on a problem with $9\%$ positives, $91\%$ is free, and the honest number is the fraction of the *remaining* error removed. Accuracy is $\pi\,\mathrm{recall}+(1-\pi)\,\mathrm{specificity}$, blind to the rare class. |
| What are the four numbers everything else is made of? | TP, FN, FP, TN — the confusion matrix (Eq. 5.67), rows true and columns predicted; the two off-diagonal entries are two *different* errors with different costs. |
| Precision or recall — which one do I want? | A question about the application, not the model: screening wants recall (few misses), a spam filter wants precision (few false alarms); the threshold trades one for the other. |
| Why the harmonic mean in $F_1$? | It is small unless *both* precision and recall are large ($F_1\le2\min$); the arithmetic mean would reward a classifier that is perfect on one and useless on the other. |
| What does an AUC of $0.99$ mean? | Pick a random positive and a random negative: with probability $0.99$ the positive gets the higher score. It depends on the ranking alone — not on the threshold, not on the class balance. |
| The threshold moved from $0.5$ to $0.1$. What changed? | Only the operating point: weights, probabilities, ranking, ROC curve and AUC are identical; every number computed from the four counts changed, and accuracy can fall while recall rises. |
| Why is the ROC curve blind to class imbalance and the PR curve not? | TPR and FPR are each normalised within one class; precision has FP in its denominator, so the proportion of positives enters it — the PR floor is the prevalence. |
| What does the MCC add to precision and recall? | It is the correlation between labels and decisions, uses all four counts, is symmetric in the classes, and is $0$ for the majority rule — the one scalar that is hard to fool by imbalance. |
| Rows or columns of the $10\times10$ matrix? | Rows are the truth: a row over its sum is the recall of that class; columns are the decisions: a column over its sum is its precision. |
| How accurate is an accuracy? | It is a binomial proportion: standard error $\sqrt{\hat p(1-\hat p)/n}$ (Eq. 2.24) — $\pm0.15\%$ on $10\,000$ images, $\pm1.7\%$ on $143$; differences inside the error bar are not differences. |
| Why do we train on the cross entropy and report accuracy? | Accuracy is a step function of the parameters with zero gradient; the cross entropy is a smooth surrogate minimised in roughly the same place — a lower cross entropy does not guarantee a higher accuracy. |

### Practicalities

Deliver your solutions (a single Jupyter notebook) in Canvas before
**Friday October 23 at midnight**. Collaboration is encouraged — list your
collaborators. The functions you write this week are the ones you will use
in part e) of Project 2 (deadline November 6); the solutions to the
exercises of weeks 41 and 42 are in the weekly material folders of the
course repository. Validation sets and cross-validation for your Project 2
network are the topic of week 44.

**Declaration of the use of large language models (LLMs).** Many of you use
ChatGPT, Claude, Copilot or similar tools when solving the weekly exercises,
and that is fine — but it has to be declared, for the weekly exercises as
well as for the projects. Add a short paragraph at the end of your notebook
(the cell below is a template) stating which tools you used and for what
(code, debugging, explanations, text), or that you did not use any. A few
sentences are enough; the full guidelines, which you must follow in the
project reports, are at
[LLM_Usage_Declaration_Guidelines.md](https://github.com/EducationalMaterialUiO/MachineLearningUiO/blob/main/LLM_Usage_Declaration_Guidelines.md). Remember that *you* are
responsible for every line you hand in: be prepared to explain it.

## Declaration of the use of LLMs

*(Replace this text with your own declaration, or write "No LLMs were used".)*

Tools used: … (for example ChatGPT, Claude, GitHub Copilot)

What they were used for: … (for example debugging the sorted pass in
exercise 1, explaining the rank formula in exercise 1c, polishing this text)

What was done without them: …